# VMA — 1882 parcel comparison on Colab T4

Upload the prepared notebook and `vma-seg-comparison.tar.gz`. Select **Runtime → Change runtime type → GPU/T4** when available. This notebook runs bounded segmentation interactively; it does not poll a worker queue or write the VMA database.

The bundle contains pinned local code (including uncommitted changes), candidate priors and structured map context. It excludes reference traces, correction snapshots and secrets. The street/water **then-now** blocks are candidates, never reference truth. Choose fixed regions before execution. For memory, each inference region needs at least two eligible tiles; small reference windows may sit inside a larger inference region. Score only the independently traced windows, and do not warm memory on reference labels. Use separate owner-traced test windows; do not tune against their results.

You authorize Google Drive access to read `epoch_010.pth` and persist outputs. No worker token or Supabase service key is needed. The four variants isolate memory and metadata; metadata guides prompt/target settings, not prose input to SAM2. A real accuracy conclusion still needs the independent references and local scoring.


In [ ]:
from google.colab import files, drive
from pathlib import Path
import hashlib, json, subprocess, sys, tarfile

# Upload the prepared source bundle, not the whole repository or an .env file.
uploaded = files.upload()
archive_name = 'vma-seg-comparison.tar.gz'
assert archive_name in uploaded, 'Upload vma-seg-comparison.tar.gz'
ROOT = Path('/content/vma-comparison')
ROOT.mkdir(exist_ok=True)
with tarfile.open(archive_name) as archive:
    for member in archive.getmembers():
        assert member.isfile() or member.isdir(), 'Links are not allowed in the bundle'
        assert not Path(member.name).is_absolute() and '..' not in Path(member.name).parts
    archive.extractall(ROOT, filter='data')
manifest = json.loads((ROOT / 'manifest.json').read_text())
for name, digest in manifest['files'].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest, 'Source hash mismatch: ' + name
print('Verified bundle:', manifest['code_revision'], list(manifest['priors']))

# The adapter was qualified against these revisions. Do not silently pull main.
for name, url, revision in [
    ('sam2', 'https://github.com/facebookresearch/sam2.git', manifest['sam2_revision']),
    ('MapSAM2', 'https://github.com/Xue-Xia/MapSAM2.git', manifest['mapsam2_revision']),
]:
    destination = ROOT / name
    if not destination.exists(): subprocess.run(['git', 'clone', '-q', url, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', revision], check=True)
assert sys.version_info >= (3, 12), 'This pinned dependency set requires Python 3.12 or newer'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests==2.34.2', 'shapely==2.2.0',
    'scipy==1.18.1', 'pillow==12.3.0', 'opencv-python-headless==4.12.0.88',
    'hydra-core==1.3.7', 'iopath==0.1.10', 'tqdm==4.70.1'], check=True)
# Keep the Colab torch/CUDA installation; verify its APIs through the preflight.
import torch
assert torch.cuda.is_available(), 'Connect a GPU runtime first'
print('GPU:', torch.cuda.get_device_name(0), 'torch:', torch.__version__)
drive.mount('/content/drive')


In [ ]:
# Explicit configuration. Empty regions prevent an accidental full-sheet run.
CHECKPOINT = Path('/content/drive/MyDrive/vma_mapsam2_cache/models/epoch_010.pth')
OUTPUT = Path('/content/drive/MyDrive/vma_seg_comparison/1882')
PRIOR_CHOICE = 'then-now' if 'then-now' in manifest['priors'] else 'colour'
IIIF_BASE = manifest['iiif_base'] or ''  # use the pinned 1882 scan URL
REGIONS = [
    # {'name': 'development', 'box': [x, y, width, height], 'split': 'development'},
    # {'name': 'test-1', 'box': [x, y, width, height], 'split': 'heldout'},
]
TILE_SIZE = 2048
RENDER_SIZE = 512  # input coarsening; SAM2 encodes internally at 1024
OVERLAP = 256

assert CHECKPOINT.is_file(), 'Place the existing epoch_010.pth checkpoint in Drive'
assert IIIF_BASE and REGIONS, 'Set the pinned IIIF URL and explicit regions before running'
assert PRIOR_CHOICE in manifest['priors']
assert all(r['split'] in ('development', 'heldout') for r in REGIONS)
assert len({r['name'] for r in REGIONS}) == len(REGIONS)
for region in REGIONS:
    x, y, width, height = region['box']
    assert x >= 0 and y >= 0 and width > 0 and height > 0
    assert x + width <= manifest['source_size'][0] and y + height <= manifest['source_size'][1]
    assert not any(x < bx + bw and bx < x + width and y < by + bh and by < y + height
        for bx, by, bw, bh in manifest['blind_boxes']), 'Region touches an unseen held-out river/road window'
    assert all(ch.isalnum() or ch in '-_' for ch in region['name']), 'Use a simple region name' 
OUTPUT.mkdir(parents=True, exist_ok=True)
import os
ENV = dict(os.environ)
ENV['PYTHONPATH'] = str(ROOT / 'sam2')
# All prompt geometry comes from local priors. No live OCR/triage or DB key.
for key in ('SUPABASE_SERVICE_KEY', 'VMA_WORKER_KEY'):
    ENV.pop(key, None)
preflight = [sys.executable, 'work/MapSAM2/inference_tiles_as_video.py', '--self-check', '--device', 'cuda',
    '--lora', '--mapsam2-dir', str(ROOT / 'MapSAM2')]
subprocess.run(preflight, cwd=ROOT, env=ENV, check=True)


In [ ]:
# Fixed 2x2 ablation: image predictor vs memory, without vs with context.
# Use a new output directory for a new frozen experiment; each result is saved
# immediately so a disconnected runtime does not lose completed runs.
variants = [('baseline', False, False), ('context', False, True),
            ('memory', True, False), ('memory-context', True, True)]
results = []
frozen_tile_inputs = {}
for region in REGIONS:
    for name, memory, context in variants:
        destination = OUTPUT / f"{region['name']}-{PRIOR_CHOICE}-{name}.json"
        assert not destination.exists(), 'Result already exists; use a new output directory'
        command = [sys.executable, 'work/MapSAM2/inference_tiles_as_video.py', '--map-id', manifest['map_id'],
            '--iiif-base', IIIF_BASE, '--checkpoint', str(CHECKPOINT), '--lora',
            '--mapsam2-dir', str(ROOT / 'MapSAM2'), '--mode', 'prompted',
            '--prior', str(ROOT / manifest['priors'][PRIOR_CHOICE]['path']), '--prior-types', 'parcel',
            '--feature-type', 'parcel', '--no-live-crop', '--region', ','.join(map(str, region['box'])),
            '--tile-size', str(TILE_SIZE), '--render-size', str(RENDER_SIZE), '--overlap', str(OVERLAP), '--device', 'cuda',
            '--out-json', str(destination)]
        if memory: command += ['--memory']
        if context: command += ['--context', str(ROOT / 'inputs/context.json')]
        # No --write-supabase, --text-mask, --watershed or test-window previews.
        log = destination.with_suffix('.log')
        with log.open('w') as stream:
            process = subprocess.Popen(command, cwd=ROOT, env=ENV, stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end=''); stream.write(line); stream.flush()
            code = process.wait()
        assert code == 0, f'Run failed; inspect {log}'
        result = json.loads(destination.read_text())
        assert result['complete'], 'Partial results cannot establish accuracy'
        prior_inputs = frozen_tile_inputs.setdefault(region['name'], result['tile_inputs'])
        assert result['tile_inputs'] == prior_inputs, 'Image pixels changed between variants; freeze the input again' 
        if memory and result['memory']['conditioned_tiles'] == 0:
            print('UNINFORMATIVE MEMORY TEST: no tile consumed memory; enlarge the region or inspect confidence.')
        results.append({'region': region, 'variant': name, 'file': str(destination),
                        'n_polys': result['n_polys'], 'memory': result['memory']})
        (OUTPUT / 'comparison-manifest.json').write_text(json.dumps({
            'bundle': manifest, 'checkpoint': str(CHECKPOINT), 'tile_size': TILE_SIZE,
            'overlap': OVERLAP, 'render_size': RENDER_SIZE, 'results': results, 'torch': torch.__version__,
            'gpu': torch.cuda.get_device_name(0)}, indent=2))
print('Saved artifacts to', OUTPUT)
print('Download JSON/logs and score locally against frozen owner traces; no reference was sent to the segmenter.')
